# SalesTeam AI — Etat d'Avancement Global du Projet

**Date de mise a jour :** 23 Juillet 2026  
**Stack :** Python 3.14 | XGBoost | FastAPI | React (Vite)  
**Depot GitHub :** https://github.com/fadizarai/salesteam_ai  

> Ce notebook est le **tableau de bord vivant** du projet. Il centralise l'architecture, les modeles IA entraines, les metriques de performance, et la TODO list globale avec le statut de chaque tache.


---
## 1. Architecture Globale du Systeme (4 Couches)

```
  FLUTTER APP (Commercial sur le terrain)
         |
         | POST /api/recommend {client_id}
         v
+----------------------------------+
|  LAYER 4 - FastAPI REST API      |
|  src/api/main.py                 |
|  routes: recommend / clients     |
|           feedback / admin       |
+----------------------------------+
         |
         v
+----------------------------------+
|  LAYER 3 - Services              |
|  recommendation.py  [FAIT]       |
|  feedback.py        [FAIT]       |
|  explanation.py     [PARTIEL]    |
+----------------------------------+
         |
         v
+----------------------------------+
|  LAYER 2 - Modeles IA            |
|  XGBClassifier  [ENTRAINE]       |
|  XGBRegressor   [ENTRAINE]       |
|  LabelEncoder   [ENTRAINE]       |
+----------------------------------+
         |
         v
+----------------------------------+
|  LAYER 1 - Data Pipeline         |
|  loader.py      [FAIT]           |
|  cleaner.py     [FAIT]           |
|  target_builder [FAIT]           |
|  feature_eng.   [FAIT]           |
+----------------------------------+
         |
         v
  data/processed/training_set.csv
  (153 320 lignes | 26 features | 737 clients | 632 articles)
```


---
## 2. Modeles IA Entraines et Metriques de Performance

### Modele 1 : XGBoost Classifier (Intention d'Achat)
> Fichier : `src/models/classifier_lsat.joblib`  
> Script  : `src/models/train_classifier.py`  
> Entraine le : 2026-07-22

| Metrique           | Valeur               |
|--------------------|----------------------|
| ROC-AUC Score      | **1.0000**           |
| PR-AUC Score       | **1.0000**           |
| F1-Score (classe 1)| **1.00**             |
| Accuracy           | **1.00**             |
| Split par client   | Oui (anti-fuite)     |
| Best Iteration     | 355 (early stopping) |
| Clients train      | 515                  |
| Clients test       | 148                  |
| scale_pos_weight   | 9.09                 |

**Top 3 Features les plus importantes :**
1. `frequency` (64.2%) — Combien de fois ce client a achete ce produit
2. `total_qty` (26.2%) — Quantite totale historiquement commandee
3. `avg_qty` (4.6%) — Quantite moyenne par commande

---

### Modele 2 : XGBoost Regressor (Quantite a Suggerer)
> Fichier : `src/models/regressor_lsat.joblib`  
> Script  : `src/models/train_regressor.py`  
> Entraine le : 2026-07-23  
> Entraine UNIQUEMENT sur les achats reels (target_qty > 0)

| Metrique          | XGBoost Regressor | Baseline avg_qty | Amelioration |
|-------------------|-------------------|------------------|--------------|
| MAE               | **2.29 unites**   | 5.86 unites      | **+61%**     |
| RMSE              | **15.5 unites**   | 26.6 unites      | **+42%**     |
| Clients train     | 348               |                  |              |
| Clients test      | 100               |                  |              |
| Best Iteration    | 112 (early stopping) |               |              |

> **Interpretation :** Le regresseur se trompe en moyenne de 2.3 unites sur la quantite
> suggeree, contre 5.9 unites pour la simple moyenne historique. Gain reel de 61% en MAE.


---
## 3. Pipeline d'Inference en Temps Reel (recommendation.py)

Le service `src/services/recommendation.py` orchestre les 2 modeles en cascade :

```
Entree : client_id = CLT009160
    |
    +-- Filtrage dataset -> ex: 128 paires (client, article)
    |
    +-- Encodage categorie via LabelEncoder (deja entraine)
    |
    +-- ETAPE 1 : XGBClassifier.predict_proba()
    |      -> [0.999, 0.999, 0.893, 0.003, ...] pour chaque article
    |      Filtre : probabilite >= 0.30
    |      Top N (configurable via le slider de l'interface React)
    |
    +-- ETAPE 2 : XGBRegressor.predict()
    |      -> [8, 21, 13, 3, 5, ...] unites a commander
    |      Clip a min=1 | arrondi entier superieur
    |      Fallback automatique : ceil(avg_qty) si regressor absent
    |
    +-- Generation explication francais (regles -> Mistral-7B futur)
    |
    v
Sortie : RecommendResponse JSON -> Flutter App / React Dashboard
```

**Status du Regressor dans recommendation.py : CONNECTE**  
- Charge depuis `src/models/regressor_lsat.joblib`  
- Fallback gracieux vers `ceil(avg_qty)` si le fichier est absent  
- La source est indiquee dans l'explication : 'IA' vs 'historique'


---
## 4. TODO LIST GLOBALE — Avancement Complet du Projet

### LAYER 1 — Data Pipeline
| # | Tache                                                 | Status   |
|---|-------------------------------------------------------|----------|
| 1 | Chargement des fichiers Excel bruts (loader.py)       | [x] FAIT |
| 2 | Nettoyage et normalisation des donnees (cleaner.py)   | [x] FAIT |
| 3 | Jointure factures + lignes + coordonnees GPS          | [x] FAIT |
| 4 | Construction de la matrice de features (26 features)  | [x] FAIT |
| 5 | Calcul de target_bought et target_qty                 | [x] FAIT |
| 6 | Gestion clients sans GPS (has_gps=False)              | [x] FAIT |
| 7 | Export CSV : data/processed/training_set.csv          | [x] FAIT |

### LAYER 2 — Modeles IA
| # | Tache                                                 | Status   |
|---|-------------------------------------------------------|----------|
| 8 | XGBoost Classifier (intention d'achat)                | [x] FAIT |
| 9 | Split anti-fuite par client (GroupShuffleSplit)       | [x] FAIT |
|10 | Early stopping sur jeu de validation                  | [x] FAIT |
|11 | Metriques ROC-AUC + PR-AUC + Matrice de Confusion     | [x] FAIT |
|12 | Feature importance Top 15                             | [x] FAIT |
|13 | Sauvegarde metadata JSON du classifieur               | [x] FAIT |
|14 | XGBoost Regressor (quantite a suggerer)               | [x] FAIT |
|15 | Regressor entraine sur achats positifs seulement      | [x] FAIT |
|16 | Comparaison MAE/RMSE vs baseline avg_qty              | [x] FAIT |
|17 | Sauvegarde metadata JSON du regresseur                | [x] FAIT |
|18 | Pipeline de retrainement auto (retrain trigger)       | [ ] TODO |

### LAYER 3 — Services
| # | Tache                                                 | Status   |
|---|-------------------------------------------------------|----------|
|19 | recommendation.py : inference classifieur XGBoost     | [x] FAIT |
|20 | recommendation.py : inference regresseur connecte     | [x] FAIT |
|21 | recommendation.py : fallback si regresseur absent     | [x] FAIT |
|22 | recommendation.py : get_available_clients()           | [x] FAIT |
|23 | feedback.py : sauvegarde CSV mensuel                  | [x] FAIT |
|24 | feedback.py : load_feedback_for_retraining()          | [x] FAIT |
|25 | explanation.py : explications basees sur regles       | [ ] TODO |
|26 | explanation.py : appel HuggingFace Mistral-7B         | [ ] TODO |
|27 | explanation.py : cache TTL 24h                        | [ ] TODO |
|28 | Service de retrainement auto hebdomadaire             | [ ] TODO |

### LAYER 4 — API Backend
| # | Tache                                                 | Status   |
|---|-------------------------------------------------------|----------|
|29 | POST /api/recommend (connecte au service)             | [x] FAIT |
|30 | GET  /api/clients (liste clients + stats)             | [x] FAIT |
|31 | POST /api/feedback (sauvegarde CSV)                   | [x] FAIT |
|32 | GET  /health + POST /api/retrain                      | [x] FAIT |
|33 | Authentification JWT (securite commerciaux)           | [ ] TODO |
|34 | Rate limiting et gestion erreurs avancee              | [ ] TODO |
|35 | Logs structures (MLflow ou Prometheus)                | [ ] TODO |

### LAYER 5 — Interface React (Frontend de Test)
| # | Tache                                                 | Status   |
|---|-------------------------------------------------------|----------|
|36 | Dashboard React avec design Glassmorphism Sombre      | [x] FAIT |
|37 | Selecteur de client (chips + recherche libre)         | [x] FAIT |
|38 | Slider pour ajuster le nombre max de suggestions      | [x] FAIT |
|39 | KPI cards (articles recommandes / confiance / %)      | [x] FAIT |
|40 | Tableau projet de commande avec barres de proba       | [x] FAIT |
|41 | Boutons Accepter / Rejeter interactifs                | [x] FAIT |
|42 | Champ quantite modifiable par le commercial           | [x] FAIT |
|43 | Bouton Soumettre feedback -> POST /api/feedback       | [ ] TODO |
|44 | Affichage info du client selectionne (panier moyen)   | [ ] TODO |
|45 | Page historique des feedbacks soumis                  | [ ] TODO |

### LAYER 6 — Production et Deploiement
| # | Tache                                                 | Status   |
|---|-------------------------------------------------------|----------|
|46 | Integration Flutter (appel POST /api/recommend)       | [ ] TODO |
|47 | Deploiement API sur serveur (VPS / Cloud)             | [ ] TODO |
|48 | Base de donnees PostgreSQL (remplace CSV feedback)    | [ ] TODO |
|49 | Pipeline CI/CD (GitHub Actions + tests auto)          | [ ] TODO |
|50 | Tunnel ngrok pour demo sur le terrain                 | [ ] TODO |


---
## 5. Chiffres Cles du Projet

| Indicateur                          | Valeur              |
|-------------------------------------|---------------------|
| Factures LSAT traitees              | ~18 444             |
| Lignes de commande nettoyees        | ~78 530             |
| Lignes dans training_set.csv        | 153 320             |
| Clients uniques                     | 737 (325 avec GPS)  |
| Articles uniques                    | 632                 |
| Features par paire client-article   | 26                  |
| Periode couverte                    | Jan 2024 - Jun 2026 |
| XGBClassifier ROC-AUC               | 1.0000              |
| XGBRegressor MAE                    | 2.29 unites         |
| Amelioration vs baseline            | +61% MAE            |
| Endpoints API actifs                | 5                   |
| **Taches completees**               | **32 / 50**         |
| **Taches restantes**                | **18 / 50**         |

---
## 6. Prochaines Etapes Prioritaires

**Priorite 1 — Cette semaine :**
- [ ] Implementer `explanation.py` (regles simples en priorite, Mistral ensuite)
- [ ] Connecter le bouton Soumettre du React vers `POST /api/feedback`

**Priorite 2 — Court terme :**
- [ ] Authentification JWT pour securiser les endpoints API
- [ ] Tunnel ngrok pour tester l'API en conditions reelles sur le terrain

**Priorite 3 — Moyen terme :**
- [ ] Integration Flutter complete
- [ ] Pipeline de retrainement auto hebdomadaire
- [ ] Deploiement sur serveur de production


---
## 7. Commandes Utiles

```powershell
# Activer l'environnement virtuel
.\venv\Scripts\Activate.ps1

# Reentrainer le Classifieur XGBoost
python src/models/train_classifier.py

# Reentrainer le Regresseur XGBoost
python src/models/train_regressor.py

# Tester l'inference CLI (Projet de Commande pour un client)
python recommend_for_client.py CLT009160

# Lancer le Backend FastAPI (port 8000)
python -m uvicorn src.api.main:app --port 8000 --reload

# Lancer l'Interface Web React
cd frontend && npm run dev
# -> http://localhost:5173/
```
